# 02 · Data profiling — Grammy Awards + link with Spotify

The notebook only reads data. Every number in the final risk table is computed by the code above it.

| ID | Analytical requirement |
|---|---|
| **AR-01** | Grammy penetration by Spotify genre |
| **AR-02** | Era of recognition: popularity and audio profile of Grammy artists by decade |
| **AR-03** | Grammy category family vs artist popularity |

**How to run**
1. `docker compose up -d mysql`, then `docker compose --profile setup run --rm source-loader` (fills the source table).
2. In the next cell, use the same MySQL password as in your `.env` (`MYSQL_ETL_PASSWORD`) and *Run All*.
3. No MySQL running? Set `FROM_MYSQL = False` to read the CSV instead. Run from the `notebooks/` folder.

In [ ]:
import os
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", None)

# --- where does the Grammy data come from? ---------------------------------------------------
FROM_MYSQL = True    # True: MySQL table grammy_source.grammy_awards   |   False: the provided CSV
MYSQL = {"host": "localhost", "port": 3307, "user": "etl_user",
         "password": os.getenv("MYSQL_ETL_PASSWORD", "change_me_etl"),   # same values as in your .env
         "database": "grammy_source"}
GRAMMY_CSV = "../data/raw/grammy/the_grammy_awards.csv"
SPOTIFY_CSV = "../data/raw/spotify/spotify_dataset.csv"
RISKS_CSV = "../docs/evidence/profiling/grammy_risk_table.csv"   # the final risk table is saved here

risks = []   # the risk table is filled while we profile


def add_risk(condition, risk_id, attribute, evidence, risk, dimension, requirement):
    """Add a row to the risk table only if the profiling really shows the problem."""
    if condition:
        risks.append({
            "Risk ID": risk_id,
            "Dataset / Attribute": attribute,
            "Profiling Evidence": evidence,
            "Potential Quality Risk": risk,
            "Quality Dimension": dimension,
            "Related Requirement": requirement,
        })


def clean_name(text):
    """Lower case, no accents, no punctuation, single spaces. Example: 'Beyoncé ' -> 'beyonce'."""
    if pd.isna(text):
        return None
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^\w\s]", " ", text.lower().replace("&", " and "))
    return re.sub(r"\s+", " ", text).strip() or None

## 1 · Load and structure
The source table keeps the CSV as raw text, so everything is a string until we convert it here, in memory.

In [ ]:
if FROM_MYSQL:
    import pymysql
    conn = pymysql.connect(**MYSQL, charset="utf8mb4")
    cursor = conn.cursor()
    cursor.execute("SELECT year, title, published_at, updated_at, category, nominee, artist, workers, img, winner "
                   "FROM grammy_awards ORDER BY source_row_id")
    grammy = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
    conn.close()
    print("Data read from MySQL: grammy_source.grammy_awards")
else:
    # same reading rules as the source loader: only an empty field is missing
    grammy = pd.read_csv(GRAMMY_CSV, dtype=str, keep_default_na=False, na_values=[""])
    print("Data read from the CSV:", GRAMMY_CSV)

print("rows x columns:", grammy.shape)
grammy.head(3)

In [ ]:
structure = pd.DataFrame({
    "non_null": grammy.notna().sum(),
    "nulls": grammy.isna().sum(),
    "% nulls": (grammy.isna().mean() * 100).round(2),
    "distinct": grammy.nunique(),
})
structure

## 2 · Completeness
`artist` is the only link with Spotify. Where is it missing?

In [ ]:
no_artist = grammy["artist"].isna()
print(f"rows without artist: {no_artist.sum():,} of {len(grammy):,} ({no_artist.mean():.1%})")

by_category = grammy.assign(no_artist=no_artist).groupby("category")["no_artist"].agg(["sum", "count"])
by_category.columns = ["rows without artist", "rows"]
by_category["% without artist"] = (by_category["rows without artist"] / by_category["rows"] * 100).round(1)
by_category = by_category.sort_values("rows without artist", ascending=False)
by_category.head(10)

In [ ]:
categories_never_artist = (by_category["% without artist"] == 100).sum()
top_missing = ", ".join(f"{name} ({int(row['rows without artist'])})" for name, row in by_category.head(3).iterrows())
workers_null = grammy["workers"].isna().mean()
img_null = grammy["img"].isna().mean()
print(f"categories where artist is always missing: {categories_never_artist} of {len(by_category)}")
print(f"workers null: {workers_null:.1%} | img null: {img_null:.1%}")

add_risk(no_artist.sum() > 0, "RK-GR-01", "artist",
         f"{no_artist.sum():,} of {len(grammy):,} rows ({no_artist.mean():.1%}) have no artist; most in {top_missing}; {categories_never_artist} categories never have one.",
         "Nominations without artist cannot be linked to Spotify: they would be left out of every KPI unless the artist can be recovered from `nominee` (see section 8).",
         "Completeness", "AR-01, AR-02, AR-03")
add_risk(workers_null >= 0.2 or img_null >= 0.2, "RK-GR-02", "workers, img",
         f"workers: {workers_null:.1%} null; img: {img_null:.1%} null (flag: 20% or more); `workers` is a free-text list.",
         "Incomplete free text: not a reliable input. Decision: leave both columns out of the Data Warehouse (they stay in the source table).",
         "Completeness", "— (informational, out of scope)")

## 3 · Duplicates

In [ ]:
exact_duplicates = grammy.duplicated().sum()
repeated_nomination = grammy.duplicated(["year", "category", "nominee", "artist"]).sum()
print("exact duplicate rows:", exact_duplicates)
print("repeated (year, category, nominee, artist):", repeated_nomination)

add_risk(exact_duplicates + repeated_nomination > 0, "RK-GR-03", "year, category, nominee, artist",
         f"{exact_duplicates:,} exact duplicate rows; {repeated_nomination:,} rows repeat (year, category, nominee, artist).",
         "Repeated nominations would be counted twice in nominations per artist, decade and category.",
         "Uniqueness", "AR-01, AR-02, AR-03")

## 4 · Winner and categories

In [ ]:
print(grammy["winner"].value_counts(dropna=False))

winner_values = grammy["winner"].nunique()
add_risk(winner_values < 2, "RK-GR-04", "winner",
         f"`winner` has {winner_values} distinct value(s): {grammy['winner'].value_counts(dropna=False).to_dict()}.",
         "If `winner` never changes, nominees cannot be told apart from winners: win_count and the Winner/Nominee status carry no information (the requirements only need 'recognized').",
         "Validity", "— (informational)")

In [ ]:
print("distinct categories:", grammy["category"].nunique())
category_clean = grammy["category"].str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
print("after trimming spaces and lower-casing:", category_clean.nunique())
extra_spaces = (grammy["category"].dropna() != grammy["category"].dropna().str.strip().str.replace(r"\s+", " ", regex=True)).sum()
print("categories written with extra spaces:", extra_spaces)

grammy["category"].value_counts().head(10)

In [ ]:
# Preview of the category groups needed by AR-03. The first group that matches wins.
GROUP_KEYWORDS = [
    ("General Field", ["record of the year", "album of the year", "song of the year", "best new artist"]),
    ("Production / Technical", ["producer", "engineer", "remix", "package", "notes", "music video", "music film"]),
    ("Classical", ["classical", "orchestral", "opera", "chamber", "choral", "conductor", "composition"]),
    ("Latin", ["latin", "tejano", "tropical", "banda", "norteno", "mexican"]),
    ("Jazz", ["jazz"]),
    ("Gospel / Christian", ["gospel", "christian"]),
    ("R&B", ["r&b", "rhythm and blues", "soul", "urban contemporary"]),
    ("Rap", ["rap", "hip hop", "hip-hop"]),
    ("Country / Folk / Blues", ["country", "bluegrass", "folk", "americana", "blues", "roots"]),
    ("Rock / Alternative", ["rock", "metal", "alternative", "punk"]),
    ("Dance / Electronic", ["dance", "electronic"]),
    ("Reggae / World", ["reggae", "world music", "global music"]),
    ("Pop", ["pop"]),
    ("Visual Media / Theater / Spoken", ["visual media", "musical theater", "soundtrack", "spoken", "comedy", "children"]),
]


def category_group(name):
    text = str(name).lower()
    for group, words in GROUP_KEYWORDS:
        if any(word in text for word in words):
            return group
    return "Other"


grammy["category_group"] = grammy["category"].map(category_group)
groups = grammy["category_group"].value_counts()
groups.sort_values().plot(kind="barh", figsize=(8, 4), title="Rows per category group (preview)")
plt.show()

other_share = (grammy["category_group"] == "Other").mean()
print(f"rows left in 'Other': {other_share:.1%}")
grammy.loc[grammy["category_group"] == "Other", "category"].value_counts().head(10)

In [ ]:
add_risk(extra_spaces > 0 or category_clean.nunique() < grammy["category"].nunique(), "RK-GR-05", "category",
         f"{grammy['category'].nunique()} distinct categories; {category_clean.nunique()} after trimming spaces and lower-casing; {extra_spaces} written with extra spaces.",
         "Spelling or spacing variants would split one category into several members of the category dimension.",
         "Consistency", "AR-03")
add_risk(other_share >= 0.01, "RK-GR-06", "category -> category_group",
         f"The keyword rules leave {other_share:.1%} of the rows in 'Other' (flag: 1% or more).",
         "AR-03 needs a category family; an 'Other' bucket that is too big cannot be interpreted until the rules are extended.",
         "Completeness", "AR-03")

## 5 · The `artist` column: several artists in one value?

In [ ]:
artists = grammy["artist"].dropna()
composite = artists.str.contains(r",|&|\band\b|feat|\bwith\b|/|\+", case=False, regex=True)
print(f"artist values with a separator (',', '&', 'and', 'feat', 'with', '/', '+'): {composite.sum():,} ({composite.mean():.1%} of the non-null artists)")
artists[composite].drop_duplicates().head(8)

In [ ]:
add_risk(composite.sum() > 0, "RK-GR-07", "artist",
         f"{composite.sum():,} of {len(artists):,} artist values ({composite.mean():.1%}) contain a separator such as ',', '&', 'and', 'feat' or '/'.",
         "One credit can name several artists, but real group names ('Earth, Wind & Fire') use the same symbols: splitting blindly breaks them, and comparing the whole text misses collaborations.",
         "Validity", "AR-01, AR-02, AR-03")

## 6 · Years and dates
AR-02 uses the year of the award to build decades.

In [ ]:
years = pd.to_numeric(grammy["year"], errors="coerce")
not_numeric = grammy["year"].notna().sum() - years.notna().sum()
print("year range:", int(years.min()), "-", int(years.max()), "| not numeric:", not_numeric)

present = set(years.dropna().astype(int))
missing_years = [y for y in range(int(years.min()), int(years.max()) + 1) if y not in present]
print("years without rows inside the range:", missing_years or "none")

rows_per_decade = (years // 10 * 10).dropna().astype(int).value_counts().sort_index()
rows_per_decade.plot(kind="bar", title="Grammy rows per decade")
plt.show()

# does the year inside `title` agree with `year`?
year_in_title = pd.to_numeric(grammy["title"].str.extract(r"\((\d{4})\)")[0], errors="coerce")
title_mismatch = (year_in_title.notna() & (year_in_title != years)).sum()
print("rows where the year inside `title` differs from `year`:", title_mismatch)

In [ ]:
# published_at / updated_at: are they award dates or just page metadata?
for col in ["published_at", "updated_at"]:
    parsed = pd.to_datetime(grammy[col], format="ISO8601", errors="coerce", utc=True)
    print(f"{col}: not parseable = {grammy[col].notna().sum() - parsed.notna().sum()} | distinct values = {grammy[col].nunique()}")
bad_dates = sum(grammy[c].notna().sum() - pd.to_datetime(grammy[c], format="ISO8601", errors="coerce", utc=True).notna().sum()
                for c in ["published_at", "updated_at"])
few_dates = max(grammy["published_at"].nunique(), grammy["updated_at"].nunique())

add_risk(not_numeric > 0 or len(missing_years) > 0 or title_mismatch > 0, "RK-GR-08", "year, title",
         f"year not numeric: {not_numeric}; years missing inside the range: {missing_years or 'none'}; year in `title` different from `year`: {title_mismatch} rows.",
         "The era analysis depends on `year`: gaps or contradictions would place nominations in the wrong decade.",
         "Consistency", "AR-02")
add_risk(bad_dates > 0 or few_dates <= 5, "RK-GR-09", "published_at, updated_at",
         f"not parseable: {bad_dates}; at most {few_dates} distinct values per column.",
         "These are publication dates of the source page, not the date of the award: they must not be used as event dates (decision: leave them out of the Data Warehouse).",
         "Timeliness", "— (informational, out of scope)")

## 7 · Cross-source: can Grammy and Spotify be linked?
The only shared attribute is the artist name. We test four ways of matching, from the strictest to the most tolerant.

In [ ]:
spotify = pd.read_csv(SPOTIFY_CSV, usecols=["track_id", "artists", "track_name"])
spotify_artists = spotify["artists"].dropna().str.split(";").explode()       # one value per credited artist
spotify_keys = set(spotify_artists.map(clean_name).dropna())                  # cleaned names
print(f"Spotify: {len(spotify):,} rows | {spotify['track_id'].nunique():,} tracks | {len(spotify_keys):,} distinct cleaned artist names")

In [ ]:
SEPARATORS = re.compile(r"\s*(?:,|;|/|&|\+|\band\b|\bwith\b|\bfeaturing\b|\bfeat\b\.?|\bft\b\.?)\s*", re.IGNORECASE)


def spotify_matches(artist, nominee=None):
    """Cleaned Spotify artist names matched by one Grammy row.
    1) the whole artist text, 2) if that fails, each piece of a composite text, 3) if there is no artist, the nominee."""
    if pd.notna(artist):
        whole = clean_name(artist)
        if whole in spotify_keys:
            return [whole]
        pieces = [clean_name(piece) for piece in SEPARATORS.split(str(artist))]
        return [p for p in pieces if p in spotify_keys]
    nominee_key = clean_name(nominee)
    return [nominee_key] if nominee_key in spotify_keys else []


total = len(grammy)
exact = grammy["artist"].isin(set(spotify_artists)).sum()
cleaned = grammy["artist"].map(clean_name).isin(spotify_keys).sum()
with_split = sum(len(spotify_matches(a)) > 0 for a in grammy["artist"])
with_nominee = sum(len(spotify_matches(a, n)) > 0 for a, n in zip(grammy["artist"], grammy["nominee"]))

ladder = pd.DataFrame({
    "strategy": ["1. exact text", "2. cleaned name (accents, case, punctuation)", "3. cleaned + split composite artists", "4. 3 + nominee when artist is empty"],
    "Grammy rows matched": [exact, cleaned, with_split, with_nominee],
})
ladder["% of all rows"] = (ladder["Grammy rows matched"] / total * 100).round(1)
print(ladder.to_string(index=False))
ladder.set_index("strategy")["% of all rows"].plot(kind="barh", figsize=(9, 3), xlim=(0, 100), title="% of Grammy rows linked to a Spotify artist")
plt.show()

In [ ]:
# How many empty-artist rows can be recovered through the nominee?
empty_artist_rows = grammy[grammy["artist"].isna()]
recovered = sum(len(spotify_matches(None, n)) > 0 for n in empty_artist_rows["nominee"])
print(f"rows without artist: {len(empty_artist_rows):,} | recovered through the nominee: {recovered:,} ({recovered / max(len(empty_artist_rows), 1):.1%})")

# most frequent Grammy artists that were NOT found in Spotify
not_found = grammy[grammy["artist"].notna() & grammy["artist"].map(lambda a: len(spotify_matches(a)) == 0)]
not_found["artist"].value_counts().head(10)

In [ ]:
add_risk(with_nominee < total, "RK-XS-01", "artist (Grammy) vs artists (Spotify)",
         f"Grammy rows linked to a Spotify artist: exact text {exact:,} ({exact / total:.1%}); cleaned name {cleaned:,} ({cleaned / total:.1%}); "
         f"+ split {with_split:,} ({with_split / total:.1%}); + nominee {with_nominee:,} ({with_nominee / total:.1%}). Unlinked: {total - with_nominee:,}.",
         "Exact comparison links too few rows: names must be cleaned and composite artists split. Some rows will never link (artist not in the Spotify file, aliases); they must be kept and counted, not dropped.",
         "Consistency", "AR-01, AR-02, AR-03")
add_risk(len(empty_artist_rows) > 0, "RK-XS-02", "artist (empty) -> nominee",
         f"Of {len(empty_artist_rows):,} rows without artist, {recovered:,} ({recovered / max(len(empty_artist_rows), 1):.1%}) can be recovered with the nominee; {len(empty_artist_rows) - recovered:,} stay unresolved.",
         "The nominee only recovers part of the missing artists and could take a work title that equals an artist name: recovered rows need a flag that says where the artist came from.",
         "Completeness", "AR-01, AR-02, AR-03")

### What if we simply joined the two files by artist?

In [ ]:
spotify_rows_per_artist = spotify_artists.map(clean_name).value_counts()      # Spotify rows credited to each cleaned name
matches = [spotify_matches(a, n) for a, n in zip(grammy["artist"], grammy["nominee"])]
grammy_rows_matched = sum(len(m) > 0 for m in matches)
naive_join_rows = sum(spotify_rows_per_artist.get(key, 0) for keys in matches for key in keys)

print(f"Spotify rows: {len(spotify):,} | Grammy rows linked: {grammy_rows_matched:,}")
print(f"rows produced by a row-by-row join through the artist: {naive_join_rows:,}")
print(f"  = {naive_join_rows / len(spotify):.1f} x the Spotify rows and {naive_join_rows / max(grammy_rows_matched, 1):.1f} x the linked Grammy rows")

add_risk(naive_join_rows > max(len(spotify), grammy_rows_matched), "RK-XS-03", "artist join cardinality",
         f"An artist has many Spotify rows and many Grammy rows (many-to-many): a row-by-row join gives {naive_join_rows:,} rows "
         f"({naive_join_rows / len(spotify):.1f}x the Spotify rows, {naive_join_rows / max(grammy_rows_matched, 1):.1f}x the linked Grammy rows).",
         "Joining row by row multiplies the rows and counts every measure several times. The sources must be linked through an artist dimension, keeping two fact tables at their own grain.",
         "Integrity", "AR-01, AR-02, AR-03")

In [ ]:
# Could songs be linked directly (Grammy nominee = Spotify track name)? Title-only check, so it is an upper limit.
spotify_titles = set(spotify["track_name"].dropna().map(clean_name))
title_hits = grammy["nominee"].map(clean_name).isin(spotify_titles).sum()
print(f"Grammy nominees whose title exists as a Spotify track name: {title_hits:,} ({title_hits / total:.1%})")

add_risk(title_hits / total < 0.5, "RK-XS-04", "nominee (Grammy) vs track_name (Spotify)",
         f"Only {title_hits:,} Grammy rows ({title_hits / total:.1%}) have a nominee that matches a Spotify track name, even checking the title alone (flag: below 50%).",
         "`nominee` is often not a song (albums, people) and titles repeat across artists, so a track-level link is weak. The integration is designed at artist level.",
         "Consistency", "AR-01, AR-02, AR-03")

In [ ]:
spotify_has_date = [c for c in pd.read_csv(SPOTIFY_CSV, nrows=0).columns if re.search(r"date|year|release|month", c, re.I)]
print("Spotify date-like columns:", spotify_has_date or "none", "| Grammy years:", int(years.min()), "-", int(years.max()))

add_risk(not spotify_has_date, "RK-XS-05", "time axis",
         f"Spotify has no date column; Grammy covers {int(years.min())}-{int(years.max())} ({len(rows_per_decade)} decades).",
         "Popularity and audio features are a snapshot of today, while the awards span six decades: the results describe the artist now, not at the time of the award, and say nothing about cause and effect.",
         "Timeliness", "AR-02, AR-03")

## 8 · Risk table (Grammy and cross-source)
Required by point 6.3: *Dataset / Attribute · Profiling Evidence · Potential Quality Risk · Related Requirement* (plus an ID and the quality dimension, which is used in point 6.5).

In [ ]:
risk_table = pd.DataFrame(risks)
risk_table

In [ ]:
Path(RISKS_CSV).parent.mkdir(parents=True, exist_ok=True)
risk_table.to_csv(RISKS_CSV, index=False)
print(f"{len(risk_table)} risks found. Saved to {RISKS_CSV}")

## How these findings are used next
* **Artist integration key (6.8):** `RK-GR-07`, `RK-XS-01`, `RK-XS-02` → clean names, split composite artists, use the nominee when the artist is empty (and keep a flag with the origin).
* **Dimensional model (6.4):** `RK-XS-03`, `RK-XS-04` → two fact tables joined only through `dim_artist`; no song-level link.
* **Quality rules and thresholds (6.5):** `RK-GR-01`, `RK-GR-03`, `RK-GR-05`, `RK-GR-08` → completeness, uniqueness and consistency rules for the raw validation gate.
* **Category groups for AR-03:** `RK-GR-05`, `RK-GR-06` → final keyword rules in the transformation step.
* **Analysis limits:** `RK-GR-04`, `RK-XS-05` → KPIs use "recognized", and results are associations, not causes.